# 152. Maximum Product Subarray
**Difficulty:** 🟡 Medium · **Topic:** Array · **LeetCode:** https://leetcode.com/problems/maximum-product-subarray/

## 💡 Concepts

**Core concept(s):** A Kadane-style sweep that tracks **both** the running max **and** the running min product.

**Why tracking two values:** With multiplication, a **negative** number flips sign — today's *smallest* (most negative) product can become the *largest* the moment we multiply by another negative. So the best product ending here could come from either the previous max or the previous min. Zeros reset both.

**Key intuition / mental model:** Carry `cur_max` and `cur_min`. At each element consider `{x, cur_max*x, cur_min*x}` — the new max and min are the largest and smallest of those three.

---

### 📚 Why min matters (sign-flip intuition)
Sum-based Kadane only needs the running max because addition preserves ordering. **Products don't:** multiplying by a negative reverses which extreme is best. Keeping the running **min** captures the "most negative" candidate that a future negative can turn into the winner.

## 📝 Problem

Return the largest product of any contiguous subarray of `nums`.

**Example**
```
Input:  nums = [2, 3, -2, 4]     Output: 6     # [2, 3]
Input:  nums = [-2, 0, -1]       Output: 0
```
**Constraints:** `1 <= len(nums) <= 2 * 10^4`.

> Two meaningfully distinct approaches are shown: an O(n²) brute force and the O(n) two-variable optimal.

### Approach 1 — Brute Force (worst)

**Idea:** For every start `i`, extend a running product over `j >= i`, tracking the max.

**Time complexity:** `O(n^2)`.

**Space complexity:** `O(1)`.

In [ ]:
from typing import List

def max_product_brute(nums: List[int]) -> int:
    best = float("-inf")
    n = len(nums)
    for i in range(n):                     # start of the subarray
        run = 1
        for j in range(i, n):              # extend it one element at a time
            run *= nums[j]                 # running PRODUCT of nums[i..j]
            best = max(best, run)          # track the largest product
    return best

### Approach 2 — Track Max & Min (optimal)

**Idea:** Sweep once keeping `cur_max` and `cur_min` of products ending here. On a negative element the two swap roles, so consider all of `{x, cur_max*x, cur_min*x}`.

**Time complexity:** `O(n)`.

**Space complexity:** `O(1)`.

In [ ]:
from typing import List

def max_product_optimal(nums: List[int]) -> int:
    best = cur_max = cur_min = nums[0]     # track BOTH the biggest and smallest running product
    for x in nums[1:]:
        # A negative x swaps big<->small, so consider all three candidates.
        cand = (x, cur_max * x, cur_min * x)
        cur_max, cur_min = max(cand), min(cand)
        best = max(best, cur_max)          # the answer is the biggest product seen
    return best

In [ ]:
# Correctness check
tests = [
    ([2, 3, -2, 4], 6),
    ([-2, 0, -1], 0),
    ([-2, 3, -4], 24),                     # two negatives multiply to a big positive
    ([-2], -2),
]
for nums, expected in tests:
    b, o = max_product_brute(nums), max_product_optimal(nums)
    print(f"{nums} -> brute={b}, optimal={o} | expected={expected}")
    assert b == o == expected, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |
| `O(n³)`       | ≈ **8×** |

Inputs are built to force the **worst case** (no early exit) so the measurement reflects the true bound. Sub-millisecond rows are noisy — look at the trend, not one number.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark   # shared: prints ratio table + optional log-log plot

def make_worst_case(n):
    # Values in {+1, -1}: no zeros (so nothing resets/short-circuits) and products stay
    # bounded, keeping each multiply O(1) so the timing reflects algorithmic shape.
    nums = [1 if i % 2 == 0 else -1 for i in range(n)]
    return (nums,)

solutions = {
    "brute   O(n^2)": max_product_brute,
    "optimal O(n)  ": max_product_optimal,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Track both extremes under sign flips:** For multiplicative running optima, carry max **and** min because a negative reverses them.
- **Kadane variant:** Same "extend or restart" skeleton as Maximum Subarray, adapted to multiplication and zeros.
- **Signal to reach for it:** "maximum product subarray", any running optimum where an operation can invert ordering (negatives, reciprocals).
- **Related problems:** Maximum Subarray, Subarray Product Less Than K, House Robber.
- **Common pitfalls:** (1) keeping only the max — fails on `[-2,3,-4]`; (2) forgetting `x` alone as a candidate (needed after a zero); (3) not seeding from `nums[0]`.